# Anomaly detection on sensor time series

Five detectors on nine labeled series from the [Numenta Anomaly Benchmark](https://github.com/numenta/NAB):
rolling z-score (baseline), Isolation Forest on window features, matrix profile discords (stumpy, z-normalized and raw)
and a 1D-conv autoencoder.

Each series is split chronologically into train (40%), validation (20%) and test (40%). Detectors are fit on train with labeled
windows removed. For series with a weekly cycle (nyc_taxi, ambient_temperature) the median time-of-week profile, estimated on
normal training points, is subtracted first. The threshold is the same score quantile on normal validation points for every detector.

Evaluation runs on test at event level: a labeled window counts as detected if at least one alarm falls inside it, each alarm run
outside a window is one false positive. Alarms within one context length after a window are ignored, because the trailing window
still contains the anomaly. Isolation Forest and autoencoder are run with five seeds.

In [ ]:
import json

import pandas as pd
import plotly.io as pio

from src.data import ROOT
from src.plots import hero, small_multiples, sweep_chart

pio.renderers.default = "notebook_connected"
RESULTS = ROOT / "results"
metrics = pd.read_csv(RESULTS / "metrics.csv")
sweep = pd.read_csv(RESULTS / "sweep.csv")
scores = pd.read_parquet(RESULTS / "scores.parquet")
meta = json.loads((RESULTS / "meta.json").read_text())
order = list(metrics["detector"].unique())

## Results

NAB windows are wide, so random alarms often land inside them. The chance F1 shifts each detector's test alarm sequence by a random
circular offset (1000 times), which keeps the number and length of alarm runs but breaks their alignment with the labels.
The p-value is the share of shifts reaching at least the observed F1.

Cells show event F1 (mean over seeds) with chance F1 in brackets; `*` marks p < 0.05.

In [ ]:
per = metrics.groupby(["series", "detector"], sort=False).agg(
    f1=("f1", "mean"), chance=("chance_f1", "mean"), p=("p_value", "median"), events=("events", "first"),
)
cell = per.apply(lambda r: f"{r.f1:.2f} ({r.chance:.2f}){'*' if r.p < 0.05 else ''}", axis=1)
table = cell.unstack("detector")[order]
table.insert(0, "test events", per["events"].unstack("detector").iloc[:, 0])
table

In [ ]:
seed_sd = metrics.groupby(["series", "detector"])["f1"].std().groupby("detector").mean()
summary = per.groupby("detector").agg(
    f1=("f1", "mean"), chance=("chance", "mean"), significant=("p", lambda p: f"{(p < 0.05).sum()}/{len(p)}"),
)
summary["f1 − chance"] = summary["f1"] - summary["chance"]
summary["seed sd"] = seed_sd
summary = summary.join(metrics.groupby("detector")[["precision", "recall", "delay_h", "point_f1"]].mean())
summary.loc[order].round(2)

Delay is measured from the labeled anomaly point, which NAB places in the middle of each window, so values between minus and plus
half a window width are expected even for alarms without real lead time.

## Threshold sensitivity

Event F1 for a range of validation quantiles. A detector whose curve stays high over several quantiles ranks anomalies well;
a sharp peak means the result depends on the threshold choice. The best quantile is chosen on test data here and is therefore optimistic.

In [ ]:
sweep_chart(sweep)

## Detections over time

Grey bands are the labeled anomaly windows, the dotted line marks the start of the test part.

In [ ]:
fig = hero(scores, meta, "machine_temperature")
fig.write_image(ROOT / "docs" / "hero.png", width=1400, height=560, scale=2)
fig

In [ ]:
for name in meta:
    small_multiples(scores, meta, name).show()